In [8]:
import pandas as pd
import numpy as np
import os
import requests
from datetime import datetime
import openpyxl

pd.set_option('display.max_rows', 500)


In [9]:


# ── 1. PATH CONFIGURATION ───────────────
BASE_DATA_DIR = "../../../data"

RAW_DIR    = os.path.join(BASE_DATA_DIR, "raw")
BRONZE_DIR = os.path.join(BASE_DATA_DIR, "bronze")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(BRONZE_DIR, exist_ok=True)

DATA_URL = (
    "https://www.data.gouv.fr/api/1/datasets/r/2e3e44de-e584-4aa2-8148-670daf5617e1"
)

path_data_raw     = os.path.join(RAW_DIR, "2017_burvot_t2_france_entiere.txt")
path_rhone_bronze = os.path.join(BRONZE_DIR, "2017-bronze_burvot_t2_rhone_69.csv")



In [10]:
# ── 2. RAW LAYER: Landing Zone ──────────────────────────────────────────────
# We keep the source file exactly as it is.
if not os.path.exists(path_data_raw):
    print(f"📥 Downloading source to RAW...")
    resp = requests.get(DATA_URL, timeout=180)
    resp.raise_for_status()
    with open(path_data_raw, "wb") as f:
        f.write(resp.content)
    print(f"✅ Saved to RAW: {path_data_raw}")
else:
    print(f"✅ Raw file already exists in {RAW_DIR}")


✅ Raw file already exists in ../../../data/raw


In [11]:
print("\n📊 RAW FILE STATS")

# Taille fichier
file_size_mb = os.path.getsize(path_data_raw) / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

# Nombre de lignes (brut)
with open(path_data_raw, "r", encoding="cp1252", errors="ignore") as f:
    n_lines = sum(1 for _ in f)

print(f"Total lines (including header): {n_lines}")

# Lecture très légère (juste pour aperçu)
df_sample = pd.read_csv(
    path_data_raw,
    sep=";",
    encoding="cp1252",
    nrows=5
)

print("\nColumns detected (raw header):")
print(df_sample.columns.tolist())

print("\nSample preview:")
display(df_sample)

# Nombre de colonnes détectées
print(f"\nDetected columns count: {len(df_sample.columns)}")


📊 RAW FILE STATS
File size: 12.84 MB
Total lines (including header): 69243

Columns detected (raw header):
['Code du département', 'Libellé du département', 'Code de la circonscription', 'Libellé de la circonscription', 'Code de la commune', 'Libellé de la commune', 'Code du b.vote', 'Inscrits', 'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs', '% Blancs/Ins', '% Blancs/Vot', 'Nuls', '% Nuls/Ins', '% Nuls/Vot', 'Exprimés', '% Exp/Ins', '% Exp/Vot', 'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp']

Sample preview:


Code du département  \
1 Ain 4 4ème circonscription 1 L'Abergement-Clémenciat 1                  598   
      5 5ème circonscription 2 L'Abergement-de-Varey   1                  209   
                             4 Ambérieu-en-Bugey       1                 1116   
                                                       2                 1128   
                                                       3                 1116   

                                                          Libellé du département  \
1 Ain 4 4ème circonscription 1 L'Abergement-Clémenciat 1                     100   
      5 5ème circonscription 2 L'Abergement-de-Varey   1                      32   
                             4 Ambérieu-en-Bugey       1                     267   
                                                       2                     286   
                                                       3                     252   

                                                         Code de la circonscription  \
1 Ain 4 4ème circonscription 1 L'Abergement-Clémenciat 1                      16,72   
      5 5ème circonscription 2 L'Abergement-de-Varey   1                      15,31   
                             4 Ambérieu-en-Bugey       1                      23,92   
                                                       2                      25,35   
                                                       3                      22,58   

                                                          Libellé de la circonscription  \
1 Ain 4 4ème circonscription 1 L'Abergement-Clémenciat 1                            498   
      5 5ème circonscription 2 L'Abergement-de-Varey   1                            177   
                             4 Ambérieu-en-Bugey       1                            849   
                                                       2                            842   
                                                       3                            864   

                                                         Code de la commune  \
1 Ain 4 4ème circonscription 1 L'Abergement-Clémenciat 1              83,28   
      5 5ème circonscription 2 L'Abergement-de-Varey   1              84,69   
                             4 Ambérieu-en-Bugey       1              76,08   
                                                       2              74,65   
                                                       3              77,42   

                                                          Libellé de la commune  \
1 Ain 4 4ème circonscription 1 L'Abergement-Clémenciat 1                     37   
      5 5ème circonscription 2 L'Abergement-de-Varey   1                     21   
                             4 Ambérieu-en-Bugey       1                     90   
                                                       2                     67   
                                                       3                     80   

                                                         Code du b.vote  \
1 Ain 4 4ème circonscription 1 L'Abergement-Clémenciat 1           6,19   
      5 5ème circonscription 2 L'Abergement-de-Varey   1          10,05   
                             4 Ambérieu-en-Bugey       1           8,06   
                                                       2           5,94   
                                                       3           7,17   

                                                         Inscrits  \
1 Ain 4 4ème circonscription 1 L'Abergement-Clémenciat 1     7,43   
      5 5ème circonscription 2 L'Abergement-de-Varey   1    11,86   
                             4 Ambérieu-en-Bugey       1    10,60   
                                                       2     7,96   
                                                       3     9,26   

                                                          Abstentions  \
1 Ain 4 4ème circonscription 1 L'Abergement-Clémenciat 1            8   
      5 5è


Detected columns count: 28


In [12]:
# ── 3. BRONZE LAYER ─────────────────────
print("\nProcessing RAW -> BRONZE...")

COMMON_FIELDS = [
    "Code du département",
    "Libellé du département",
    "Code de la circonscription",
    "Libellé de la circonscription",
    "Code de la commune",
    "Libellé de la commune",
    "Code du b.vote",
    "Inscrits",
    "Abstentions",
    "% Abs/Ins",
    "Votants",
    "% Vot/Ins",
    "Blancs",
    "% Blancs/Ins",
    "% Blancs/Vot",
    "Nuls",
    "% Nuls/Ins",
    "% Nuls/Vot",
    "Exprimés",
    "% Exp/Ins",
    "% Exp/Vot",
]

CAND_FIELDS = ["N°Panneau", "Sexe", "Nom", "Prénom", "Voix", "% Voix/Ins", "% Voix/Exp"]
N_CANDIDATES = 2

all_columns = COMMON_FIELDS.copy()
for k in range(1, N_CANDIDATES + 1):
    for field in CAND_FIELDS:
        all_columns.append(f"{field}_{k}")

df_all = pd.read_csv(
    path_data_raw,
    sep=";",
    dtype=str,
    encoding="cp1252",
    header=None,
    skiprows=1,          # on saute l'en-tête du fichier source
    names=all_columns,   # on impose la vraie structure
    engine="python"
)

# nettoyage
df_all.columns = df_all.columns.str.strip()
df_all["Code du département"] = df_all["Code du département"].astype(str).str.strip().str.zfill(2)

# filtre Rhône
df_bronze = df_all[df_all["Code du département"] == "69"].copy()

# métadonnées bronze
df_bronze["extraction_source_url"] = DATA_URL
df_bronze["ingestion_timestamp"] = datetime.now().isoformat()
df_bronze["source_file_name"] = os.path.basename(path_data_raw)

# sauvegarde
df_bronze.to_csv(path_rhone_bronze, index=False, sep=";", encoding="utf-8")

display(df_bronze.head())
print(f"BRONZE dataset created: {path_rhone_bronze}")
print(f"Rows: {len(df_bronze)} | Columns: {len(df_bronze.columns)}")


Processing RAW -> BRONZE...


,Code du département,Libellé du département,Code de la circonscription,Libellé de la circonscription,Code de la commune,Libellé de la commune,Code du b.vote,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,Blancs,% Blancs/Ins,% Blancs/Vot,Nuls,% Nuls/Ins,% Nuls/Vot,Exprimés,% Exp/Ins,% Exp/Vot,N°Panneau_1,Sexe_1,Nom_1,Prénom_1,Voix_1,% Voix/Ins_1,% Voix/Exp_1,N°Panneau_2,Sexe_2,Nom_2,Prénom_2,Voix_2,% Voix/Ins_2,% Voix/Exp_2,extraction_source_url,ingestion_timestamp,source_file_name
47091,69,Rhône,08,8ème circonscription,001,Affoux,0001,257,39,"15,18",218,"84,82",20,"7,78","9,17",7,"2,72","3,21",191,"74,32","87,61",1,M,MACRON,Emmanuel,93,"36,19","48,69",2,F,LE PEN,Marine,98,"38,13","51,31",https://www.data.gouv.fr/api/1/datasets/r/2e3e...,2026-03-18T14:27:53.351361,2017_burvot_t2_france_entiere.txt
47092,69,Rhône,09,9ème circonscription,002,Aigueperse,0001,224,51,"22,77",173,"77,23",13,"5,80","7,51",11,"4,91","6,36",149,"66,52","86,13",1,M,MACRON,Emmanuel,84,"37,50","56,38",2,F,LE PEN,Marine,65,"29,02","43,62",https://www.data.gouv.fr/api/1/datasets/r/2e3e...,2026-03-18T14:27:53.351361,2017_burvot_t2_france_entiere.txt
47093,69,Rhône,05,5ème circonscription,003,Albigny-sur-Saône,0001,884,208,"23,53",676,"76,47",64,"7,24","9,47",15,"1,70","2,22",597,"67,53","88,31",1,M,MACRON,Emmanuel,433,"48,98","72,53",2,F,LE PEN,Marine,164,"18,55","27,47",https://www.data.gouv.fr/api/1/datasets/r/2e3e...,2026-03-18T14:27:53.351361,2017_burvot_t2_france_entiere.txt
47094,69,Rhône,05,5ème circonscription,003,Albigny-sur-Saône,0002,781,194,"24,84",587,"75,16",46,"5,89","7,84",11,"1,41","1,87",530,"67,86","90,29",1,M,MACRON,Emmanuel,342,"43,79","64,53",2,F,LE PEN,Marine,188,"24,07","35,47",https://www.data.gouv.fr/api/1/datasets/r/2e3e...,2026-03-18T14:27:53.351361,2017_burvot_t2_france_entiere.txt
47095,69,Rhône,09,9ème circonscription,004,Alix,0001,455,95,"20,88",360,"79,12",42,"9,23","11,67",4,"0,88","1,11",314,"69,01","87,22",1,M,MACRON,Emmanuel,204,"44,84","64,97",2,F,LE PEN,Marine,110,"24,18","35,03",https://www.data.gouv.fr/api/1/datasets/r/2e3e...,2026-03-18T14:27:53.351361,2017_burvot_t2_france_entiere.txt


BRONZE dataset created: ../../../data/bronze/2017-bronze_burvot_t2_rhone_69.csv
Rows: 1287 | Columns: 38


In [13]:
print("\n📊 BRONZE DATASET STATS")

# Taille fichier
file_size_mb = os.path.getsize(path_rhone_bronze) / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

# Shape
print(f"Rows: {df_bronze.shape[0]}")
print(f"Columns: {df_bronze.shape[1]}")

# Colonnes
print("\nColumns:")
print(df_bronze.columns.tolist())

# Types
print("\nData types:")
print(df_bronze.dtypes)

# Valeurs manquantes
print("\nMissing values per column:")
print(df_bronze.isna().sum())

# Vérification département
print("\nDepartment distribution:")
print(df_bronze["Code du département"].value_counts())

# Vérification communes (top 10)
print("\nTop 10 communes:")
print(df_bronze["Libellé de la commune"].value_counts().head(10))

# Stats sur une variable clé
print("\nInscrits stats:")
print(df_bronze["Inscrits"].astype(float).describe())


📊 BRONZE DATASET STATS
File size: 0.41 MB
Rows: 1287
Columns: 38

Columns:
['Code du département', 'Libellé du département', 'Code de la circonscription', 'Libellé de la circonscription', 'Code de la commune', 'Libellé de la commune', 'Code du b.vote', 'Inscrits', 'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs', '% Blancs/Ins', '% Blancs/Vot', 'Nuls', '% Nuls/Ins', '% Nuls/Vot', 'Exprimés', '% Exp/Ins', '% Exp/Vot', 'N°Panneau_1', 'Sexe_1', 'Nom_1', 'Prénom_1', 'Voix_1', '% Voix/Ins_1', '% Voix/Exp_1', 'N°Panneau_2', 'Sexe_2', 'Nom_2', 'Prénom_2', 'Voix_2', '% Voix/Ins_2', '% Voix/Exp_2', 'extraction_source_url', 'ingestion_timestamp', 'source_file_name']

Data types:
Code du département              object
Libellé du département           object
Code de la circonscription       object
Libellé de la circonscription    object
Code de la commune               object
Libellé de la commune            object
Code du b.vote                   object
Inscrits                     

In [14]:
df_bronze.dtypes

pd.set_option('display.max_columns', None)

display(df_bronze.head())


,Code du département,Libellé du département,Code de la circonscription,Libellé de la circonscription,Code de la commune,Libellé de la commune,Code du b.vote,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,Blancs,% Blancs/Ins,% Blancs/Vot,Nuls,% Nuls/Ins,% Nuls/Vot,Exprimés,% Exp/Ins,% Exp/Vot,N°Panneau_1,Sexe_1,Nom_1,Prénom_1,Voix_1,% Voix/Ins_1,% Voix/Exp_1,N°Panneau_2,Sexe_2,Nom_2,Prénom_2,Voix_2,% Voix/Ins_2,% Voix/Exp_2,extraction_source_url,ingestion_timestamp,source_file_name
47091,69,Rhône,08,8ème circonscription,001,Affoux,0001,257,39,"15,18",218,"84,82",20,"7,78","9,17",7,"2,72","3,21",191,"74,32","87,61",1,M,MACRON,Emmanuel,93,"36,19","48,69",2,F,LE PEN,Marine,98,"38,13","51,31",https://www.data.gouv.fr/api/1/datasets/r/2e3e...,2026-03-18T14:27:53.351361,2017_burvot_t2_france_entiere.txt
47092,69,Rhône,09,9ème circonscription,002,Aigueperse,0001,224,51,"22,77",173,"77,23",13,"5,80","7,51",11,"4,91","6,36",149,"66,52","86,13",1,M,MACRON,Emmanuel,84,"37,50","56,38",2,F,LE PEN,Marine,65,"29,02","43,62",https://www.data.gouv.fr/api/1/datasets/r/2e3e...,2026-03-18T14:27:53.351361,2017_burvot_t2_france_entiere.txt
47093,69,Rhône,05,5ème circonscription,003,Albigny-sur-Saône,0001,884,208,"23,53",676,"76,47",64,"7,24","9,47",15,"1,70","2,22",597,"67,53","88,31",1,M,MACRON,Emmanuel,433,"48,98","72,53",2,F,LE PEN,Marine,164,"18,55","27,47",https://www.data.gouv.fr/api/1/datasets/r/2e3e...,2026-03-18T14:27:53.351361,2017_burvot_t2_france_entiere.txt
47094,69,Rhône,05,5ème circonscription,003,Albigny-sur-Saône,0002,781,194,"24,84",587,"75,16",46,"5,89","7,84",11,"1,41","1,87",530,"67,86","90,29",1,M,MACRON,Emmanuel,342,"43,79","64,53",2,F,LE PEN,Marine,188,"24,07","35,47",https://www.data.gouv.fr/api/1/datasets/r/2e3e...,2026-03-18T14:27:53.351361,2017_burvot_t2_france_entiere.txt
47095,69,Rhône,09,9ème circonscription,004,Alix,0001,455,95,"20,88",360,"79,12",42,"9,23","11,67",4,"0,88","1,11",314,"69,01","87,22",1,M,MACRON,Emmanuel,204,"44,84","64,97",2,F,LE PEN,Marine,110,"24,18","35,03",https://www.data.gouv.fr/api/1/datasets/r/2e3e...,2026-03-18T14:27:53.351361,2017_burvot_t2_france_entiere.txt
